# 11. 모폴로지와 그래디언트

침식·팽창·열기·닫기와 Sobel·Laplacian 그래디언트를 실습합니다.

> 예제 이미지는 노트북 옆 `data` 폴더에 넣으세요.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_image(name, flags=cv2.IMREAD_COLOR):
    path = Path("data") / name
    image = cv2.imread(str(path), flags)
    if image is None:
        raise FileNotFoundError(path)
    return image

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()


## 테스트용 이진 영상과 구조 요소

In [ ]:
binary = np.zeros((240, 640), dtype=np.uint8)
cv2.putText(binary, "ABCDE", (20, 170), cv2.FONT_HERSHEY_SIMPLEX, 3.2, 255, 12, cv2.LINE_AA)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
show([binary, kernel * 255], ["binary image", "3x3 kernel"])


## 침식과 팽창

In [ ]:
eroded = cv2.erode(binary, kernel, iterations=2)
dilated = cv2.dilate(binary, kernel, iterations=2)
show([binary, eroded, dilated], ["source", "erosion", "dilation"])


## 열기와 닫기

In [ ]:
rng = np.random.default_rng(7)
salt = binary.copy()
ys = rng.integers(0, salt.shape[0], 1200)
xs = rng.integers(0, salt.shape[1], 1200)
salt[ys, xs] = 255

pepper = binary.copy()
ys = rng.integers(0, pepper.shape[0], 1200)
xs = rng.integers(0, pepper.shape[1], 1200)
pepper[ys, xs] = 0

opened = cv2.morphologyEx(salt, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(pepper, cv2.MORPH_CLOSE, kernel)
show([salt, opened, pepper, closed], ["salt noise", "opening", "pepper holes", "closing"])


## 형태학적 그래디언트

In [ ]:
morph_gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel)
show([binary, morph_gradient], ["source", "morphological gradient"])


## Sobel과 Laplacian

In [ ]:
gray = read_image("sudoku.jpg", cv2.IMREAD_GRAYSCALE)
gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
magnitude = cv2.magnitude(gx, gy)
laplacian = cv2.Laplacian(gray, cv2.CV_32F)

gx_view = cv2.convertScaleAbs(gx)
gy_view = cv2.convertScaleAbs(gy)
mag_view = cv2.convertScaleAbs(magnitude)
lap_view = cv2.convertScaleAbs(laplacian)
show([gray, gx_view, gy_view, mag_view, lap_view], ["source", "Sobel x", "Sobel y", "magnitude", "Laplacian"])
